# ASAP8 Detection of Change analysis

Minimal setup for cross-session Detection-of-Change analyses. This notebook deliberately stops before plotting or inferential analysis.

Canonical tables created here:
- `sessions`: one row per complete processed session
- `rois`: one row per session-specific ROI, including longitudinal identity and depth group
- `spikes`: canonical `template_v1` optical-spike table
- `events`: one row per expected image cycle, with independent change/omission marker times
- `trial_index`: lightweight map from single-trial H5 rows onto `events`

Mean-response, sequence-response, single-trial, and running loaders are imported for the analysis cells that follow.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.dataset import (
    DEPTH_GROUP_ORDER,
    build_voltage_session_table,
    build_voltage_roi_table,
)
from vip_slap2_analysis.voltage.spikes import (
    DETECTOR_VERSION,
    build_spike_table,
)
from vip_slap2_analysis.behavior.change_detection import build_change_detection_events
from vip_slap2_analysis.voltage.responses import (
    build_single_trial_index,
    load_response_package,
    get_mean_response,
    get_sequence_response,
    load_single_trial_traces,
)
from vip_slap2_analysis.behavior.encoder import compute_encoder_velocity

assert DETECTOR_VERSION == "template_v1"

## Configuration

In [ ]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")

TARGET_MICE = [852835, 863774]
TARGET_SESSION_LABELS = None
TARGET_SESSION_IDS = None
PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]

TRACE_VARIANT = "dff_robust_f0_trial"
REGISTRATION_FILENAME = "roi_identity_registration.csv"
EXCLUDE_INVALID_ROIS = True
EXPECTED_F0_SMOOTH_SEC = 60.0

FORCE_SPIKE_RECOMPUTE = False
SPIKE_KWARGS = dict(
    height_sigma=3.0,
    template_sigma=3.5,
    prominence_sigma=0.5,
)

RUNNING_KWARGS = dict(
    wheel_radius_cm=4.69,
    encoder_units="ticks",
    ticks_per_revolution=8192,
    absolute_velocity=True,
)

## Session registry

Only sessions with the full-session trace, single-trial H5, mean NPZ, sequence NPZ, corrected Bonsai log, and imaging-epoch QC are admitted. Paths to the encoder are retained when available but are not required at this stage.

In [ ]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)

sessions = build_voltage_session_table(
    registry,
    subject_ids=TARGET_MICE,
    paradigms=PARADIGMS,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
    session_labels=TARGET_SESSION_LABELS,
    session_ids=TARGET_SESSION_IDS,
    trace_variant=TRACE_VARIANT,
    expected_f0_smooth_sec=EXPECTED_F0_SMOOTH_SEC,
)

display(sessions[[
    "subject_id", "session_id", "session_label", "session_order",
    "dmd1_depth_um", "dmd2_depth_um", "f0_smooth_sec",
    "trace_h5", "single_trial_h5", "encoder_pkl",
]])

## ROI registry

The processed trace-H5 QC mask and manual registration QC are kept separately. `cell_id` uses the longitudinal `global_cell_id` when available and otherwise remains session-specific. Depth groups match the ephys notebook: `<100 µm`, `100–150 µm`, and `>150 µm`.

In [ ]:
rois = build_voltage_roi_table(
    sessions,
    registration_filename=REGISTRATION_FILENAME,
    exclude_invalid_rois=EXCLUDE_INVALID_ROIS,
)

print(f"{rois['included'].sum()} / {len(rois)} ROI observations included")
display(rois)

## Spike extraction

Uses the same source-level `template_v1` detector as the ephys notebook. Each session is cached beside its processed voltage trace as `spikes_template_v1.parquet` (compressed CSV fallback if parquet support is unavailable), with metadata that invalidates the cache if the source H5, ROI selection, detector version, or detector parameters change.

In [ ]:
spikes = build_spike_table(
    sessions,
    rois,
    force=FORCE_SPIKE_RECOMPUTE,
    detection_kwargs=SPIKE_KWARGS,
)

print(f"{len(spikes):,} spikes")
display(
    spikes.groupby(
        ["subject_id", "session_label", "dmd", "roi", "depth_group"],
        observed=True,
    ).size().rename("n_spikes").reset_index()
)

## Detection-of-Change event table

One row corresponds to one expected image cycle. `onset_sec` is the image-cycle timestamp used for image extraction. `change_onset_sec` and `omission_onset_sec` retain the special-event marker timestamps used for the longer change/omission extraction windows. This matters because a `ChangeFlash` marker can precede the changed-to image by one display frame.

The table also records expected versus actually presented sequence position, neighboring presented-image events, minute-scale time blocks, imaging epoch, and whether the complete image/change/omission extraction window was retained.

In [ ]:
events = build_change_detection_events(sessions)

summary = (
    events.groupby(["subject_id", "session_label"])
    .agg(
        n_cycles=("event_id", "size"),
        n_changes=("is_change", "sum"),
        n_omissions=("is_omission", "sum"),
        retained_images=("retained_image_window", "sum"),
        retained_changes=("retained_change_window", "sum"),
        retained_omissions=("retained_omission_window", "sum"),
    )
    .reset_index()
)

display(summary)
display(events.head())

## Single-trial index

This reads only H5 metadata/onset vectors, not the trial traces themselves. Every stored image/change/omission trial is mapped one-to-one onto the canonical event table. A mismatch raises immediately rather than being repaired downstream with repeated nearest-onset joins.

In [ ]:
trial_index = build_single_trial_index(sessions, events)

print(f"{len(trial_index):,} indexed single-trial response rows")
display(
    trial_index.groupby(
        ["subject_id", "session_label", "dmd", "event_type"]
    ).size().rename("n_trials").reset_index()
)
display(trial_index.head())

## Ready for response analyses

The base is intentionally complete at this point. Later cells can load only what they need:

```python
session = sessions.iloc[0]
mean_pkg = load_response_package(session.mean_npz)
sequence_pkg = load_response_package(session.sequence_npz)

# Mean response for one source ROI/image
# t, y = get_mean_response(mean_pkg, dmd=1, source_roi=0,
#                          event_type="image", image_name=image_name)

# Sequence-position responses for one source ROI/image
# sequence = get_sequence_response(sequence_pkg, dmd=1, source_roi=0,
#                                  image_name=image_name, phase="repeated")

# Selected single trials only
# single = load_single_trial_traces(session.single_trial_h5, dmd=1,
#                                   event_type="omission", source_rois=[0])

# HARP-aligned running
# running = compute_encoder_velocity(session.encoder_pkl, **RUNNING_KWARGS)
```

The next analysis section should begin with image-entrained response motifs, using mean dF/F as the primary response-shape representation and the spike table as a complementary fast-output readout.